# 01 — BigSolDB 2.0 audit

Milestone-0 data reality check.
Reports basic shape, unique counts, missing values, duplicates, target/temperature distributions, measurements-per-pair statistics, and temperature-span-per-pair statistics.

Writes a Markdown report to `../results/audit_report.md`. Does not modify the CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path.cwd().parent / 'data' / 'BigSolDBv2.0.csv'
OUT  = Path.cwd().parent / 'results' / 'audit_report.md'
print('Data file:', DATA)

## Load data

Expected columns per Zenodo record listed for integrity check.

In [ ]:
df = pd.read_csv(DATA)
expected = {
    'SMILES_Solute', 'Temperature_K', 'Solvent', 'SMILES_Solvent',
    'Solubility(mole_fraction)', 'Solubility(mol/L)', 'LogS(mol/L)',
    'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source',
}
missing_cols = expected - set(df.columns)
extra_cols   = set(df.columns) - expected
n_rows = len(df)
print(f'rows: {n_rows:,}')
print(f'missing_cols: {sorted(missing_cols)}')
print(f'extra_cols:   {sorted(extra_cols)}')

## Unique counts

In [ ]:
n_solutes  = df['SMILES_Solute'].nunique()
n_solvents = df['SMILES_Solvent'].nunique()
pair_key   = list(zip(df['SMILES_Solute'], df['SMILES_Solvent']))
n_pairs    = len(set(pair_key))
n_sources  = int(df['Source'].nunique()) if 'Source' in df.columns else None

print(f'unique solutes:  {n_solutes:,}')
print(f'unique solvents: {n_solvents:,}')
print(f'unique (solute, solvent) pairs: {n_pairs:,}')
print(f'unique source articles: {n_sources:,}')

if 'FDA_Approved' in df.columns:
    fda = df['FDA_Approved'].astype(str).str.lower().isin({'true','1','yes','y'}).sum()
    print(f'FDA-approved solute rows: {int(fda):,}')

## Missing values, duplicates, target and temperature distributions

In [ ]:
null_counts = df.isna().sum().sort_values(ascending=False)
print('columns with missing values:')
print(null_counts[null_counts > 0].to_string())
print()

full_dup = int(df.duplicated().sum())
key_dup = int(df.duplicated(subset=['SMILES_Solute','SMILES_Solvent','Temperature_K']).sum())
print(f'full-row duplicates: {full_dup:,}')
print(f'duplicates on (SMILES_Solute, SMILES_Solvent, Temperature_K): {key_dup:,}')
print('  (these are legitimate replicate measurements defining the aleatoric floor)')

In [ ]:
target = df['LogS(mol/L)']
print('LogS(mol/L) distribution')
print(target.describe().to_string())

T = df['Temperature_K']
print('\nTemperature_K distribution')
print(T.describe().to_string())
print(f'\nn_unique temperatures: {T.nunique():,}')

## Measurements per (solute, solvent) pair

In [ ]:
pair_group = df.groupby(['SMILES_Solute','SMILES_Solvent'])
n_meas_per_pair = pair_group.size()

thresholds = [1, 2, 3, 5, 10, 20]
for t in thresholds:
    print(f'pairs with >= {t:>3} measurements: {(n_meas_per_pair >= t).sum():,}')
print()
print(f'median measurements/pair: {int(n_meas_per_pair.median())}')
print(f'mean measurements/pair:   {n_meas_per_pair.mean():.2f}')
print(f'max measurements/pair:    {int(n_meas_per_pair.max())}')

## Temperature span (ΔT = Tmax − Tmin) per pair

In [ ]:
T_min_per_pair = pair_group['Temperature_K'].min()
T_max_per_pair = pair_group['Temperature_K'].max()
delta_T_per_pair = T_max_per_pair - T_min_per_pair

print(f'pairs with only one T (ΔT=0): {int((delta_T_per_pair == 0).sum()):,}')
for t in [10, 20, 40, 60, 80]:
    print(f'pairs with ΔT >= {t:>2}K: {(delta_T_per_pair >= t).sum():,}')
print()
print(f'median ΔT among multi-T pairs: {delta_T_per_pair[delta_T_per_pair > 0].median():.1f} K')
print(f'max ΔT: {delta_T_per_pair.max():.1f} K')

## Feasibility: pairs meeting joint conditions (T-extrapolation setup)

In [ ]:
def pairs_meeting(min_meas, min_dT):
    ok = (n_meas_per_pair >= min_meas) & (delta_T_per_pair >= min_dT)
    return int(ok.sum())

for min_m, min_dT in [(3, 20), (5, 20), (5, 40), (10, 40), (10, 60)]:
    print(f'pairs with >= {min_m} meas AND ΔT >= {min_dT}K: '
          f'{pairs_meeting(min_m, min_dT):,}')

## Top solvents and solutes by number of measurements

In [ ]:
solvent_counts = df['SMILES_Solvent'].value_counts()
solute_counts  = df['SMILES_Solute'].value_counts()
print('top 15 solvents:')
print(solvent_counts.head(15).to_string())

In [ ]:
print('top 10 solutes:')
print(solute_counts.head(10).to_string())

## Notes

- Aleatoric floor (from Krasnov et al.): RMSE ≈ 0.39 logS across labs.
- T-extrapolation feasibility: driven by the joint-condition table above.
- Sanity check on primary claims: row count should be ≈ 103,944; solutes ≈ 1,448; solvents ≈ 213 per Krasnov et al. 2025.